In [1]:
"""
HomeGuard Security System Simulator
Author: Ellen Sea
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""
 
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

In [2]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {
        "sensor_id": sensor_id,
        "location": location,
        "sensor_type": sensor_type,
        "threshold": threshold
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

# Initialize sensors for the Peterson home
sensors = [
    create_sensor("S1", "Living Room", "motion"),
    create_sensor("S2", "Kitchen",     "temperature", threshold=75),
    create_sensor("S3", "Front Door",  "door"),
    create_sensor("S4", "Bedroom",     "smoke")
]

In [3]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['sensor_id']}: {sensor['location']} ({sensor['sensor_type'].capitalize()})")

Initialized 4 sensors
  - S1: Living Room (Motion)
  - S2: Kitchen (Temperature)
  - S3: Front Door (Door)
  - S4: Bedroom (Smoke)


In [4]:
def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["sensor_type"]

    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value == True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    else:
        return False


def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered.

    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)

    Returns:
    - True if a security alert should be triggered, False otherwise
    """
    sensor_type = sensor["sensor_type"]

    # Security alerts only matter when away or sleeping
    if system_mode == "HOME":
        return False

    if system_mode == "AWAY":
        if sensor_type == "motion" and reading_value == True:
            return True
        if sensor_type == "door" and reading_value == "OPEN":
            return True

    if system_mode == "SLEEP":
        if sensor_type == "door" and reading_value == "OPEN":
            return True

    return False

In [5]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [6]:
def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_id = sensor["sensor_id"]
    location  = sensor["location"]

    # Security alerts (motion/door in AWAY mode)
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        alerts.append(create_alert(
            "HIGH",
            f"Security alert: {sensor['sensor_type'].capitalize()} triggered in {location}!",
            sensor_id,
            timestamp
        ))

    # Safety alerts (always active)
    if sensor["sensor_type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert(
            "CRITICAL",
            f"Smoke detected in {location}! Evacuate immediately!",
            sensor_id,
            timestamp
        ))

    if sensor["sensor_type"] == "temperature":
        if reading_value >= 95:
            alerts.append(create_alert(
                "CRITICAL",
                f"Dangerously high temperature ({reading_value}°F) in {location}!",
                sensor_id,
                timestamp
            ))
        elif reading_value <= 35:
            alerts.append(create_alert(
                "HIGH",
                f"Dangerously low temperature ({reading_value}°F) in {location}!",
                sensor_id,
                timestamp
            ))

    # Comfort notifications (HOME mode only)
    if system_mode == "HOME" and sensor["sensor_type"] == "temperature":
        if 36 <= reading_value <= 60:
            alerts.append(create_alert(
                "LOW",
                f"Temperature ({reading_value}°F) in {location} is below comfort range.",
                sensor_id,
                timestamp
            ))
        elif 80 <= reading_value <= 89:
            alerts.append(create_alert(
                "MEDIUM",
                f"Temperature ({reading_value}°F) in {location} is above comfort range.",
                sensor_id,
                timestamp
            ))

    return alerts

In [9]:
def generate_reading(sensor):
    sensor_type = sensor["sensor_type"]
    if sensor_type == "temperature":
        return random.randint(30, 100)
    elif sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])

def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_id = sensor["sensor_id"]
    location = sensor["location"]
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        alerts.append(create_alert("HIGH", f"Security alert: {sensor['sensor_type'].capitalize()} triggered in {location}!", sensor_id, timestamp))
    if sensor["sensor_type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("CRITICAL", f"Smoke detected in {location}! Evacuate immediately!", sensor_id, timestamp))
    if sensor["sensor_type"] == "temperature":
        if reading_value >= 95:
            alerts.append(create_alert("CRITICAL", f"Dangerously high temperature ({reading_value}F) in {location}!", sensor_id, timestamp))
        elif reading_value <= 35:
            alerts.append(create_alert("HIGH", f"Dangerously low temperature ({reading_value}F) in {location}!", sensor_id, timestamp))
    if system_mode == "HOME" and sensor["sensor_type"] == "temperature":
        if 36 <= reading_value <= 60:
            alerts.append(create_alert("LOW", f"Temperature ({reading_value}F) in {location} is below comfort range.", sensor_id, timestamp))
        elif 80 <= reading_value <= 89:
            alerts.append(create_alert("MEDIUM", f"Temperature ({reading_value}F) in {location} is above comfort range.", sensor_id, timestamp))
    return alerts

def trigger_alert(alert):
    severity_symbol = {"LOW": "ℹ️", "MEDIUM": "⚠️", "HIGH": "🚨", "CRITICAL": "🔥"}
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [10]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: False
[ALERT!] 🚨 HIGH: Security alert: Motion triggered in Living Room!


In [11]:
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """

    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id            = sensor_id
        self.location      = location
        self.sensor_type   = sensor_type
        self.threshold     = threshold
        self.current_value = None

    def read(self):
        """
        Generates and stores a new reading for this sensor.

        Returns:
        - The reading value
        """
        if self.sensor_type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.sensor_type == "motion":
            self.current_value = random.choice([True, False])
        elif self.sensor_type == "door":
            self.current_value = random.choice(["OPEN", "CLOSED"])
        elif self.sensor_type == "smoke":
            self.current_value = random.choice(["CLEAR", "DETECTED"])
        return self.current_value

    def isAbnormal(self):
        """
        Checks if the current reading is abnormal.

        Returns:
        - True if the reading is abnormal, False otherwise
        """
        if self.current_value is None:
            return False
        if self.sensor_type == "temperature":
            return self.current_value < 35 or self.current_value > 95
        elif self.sensor_type == "motion":
            return self.current_value == True
        elif self.sensor_type == "door":
            return self.current_value == "OPEN"
        elif self.sensor_type == "smoke":
            return self.current_value == "DETECTED"
        return False

    def reset(self):
        """
        Resets the sensor's current reading to None.
        """
        self.current_value = None

    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"


# Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001",   "Kitchen",     "temperature", threshold=35),
    Sensor("DOOR_001",   "Front Door",  "door"),
    Sensor("SMOKE_001",  "Bedroom",     "smoke")
]

In [12]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 61
Is abnormal: False
Sensor info: TEST_001 (Test Room): 61


In [13]:
import time

def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.

    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room",  "motion"),
        Sensor("TEMP_001",   "Kitchen",      "temperature", threshold=35),
        Sensor("DOOR_001",   "Front Door",   "door"),
        Sensor("SMOKE_001",  "Bedroom",      "smoke")
    ]

    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        for sensor in sensors:
            reading = sensor.read()

            # Display the reading
            if sensor.sensor_type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.sensor_type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.sensor_type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.sensor_type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

            # Convert sensor object to dict for process_reading
            sensor_dict = {
                "sensor_id": sensor.id,
                "location":  sensor.location,
                "sensor_type": sensor.sensor_type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)

            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

        time.sleep(0.5)

    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)


# Run the simulation
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 17:20:18
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Security alert: Motion triggered in Living Room!
[LOG] [17:20:18] Sending notification to homeowner...
[READING] Kitchen Temperature: 78°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 17:20:19
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Security alert: Motion triggered in Living Room!
[LOG] [17:20:19] Sending notification to homeowner...
[READING] Kitchen Temperature: 35°F (Abnormal)
[ALERT!] 🚨 HIGH: Dangerously low temperature (35F) in Kitchen!
[LOG] [17:20:19] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 17:20:19
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!
